In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm  # Uses standard fallback progress bar to avoid ipywidgets issues

# Folder configuration anchors
PAPERS_REPO_DIR = "/Users/salehalizade/Desktop/AXIOM/paperlists"
OUTPUT_JSONL_PATH = "axiom_nlp_comprehensive_dataset.jsonl"

# Domain Focus: Pure NLP/Computational Linguistics tracks across all historical archives
NLP_CONFERENCES = ["acl", "emnlp", "naacl", "coling"]

# Administrative and structural artifact filters to prevent data corruption
ADMIN_BLACKLIST = [
    "preface", "table of contents", "acknowledgment", "organisation", 
    "organization", "committee", "session", "index", "introduction by",
    "volume", "proceedings of", "reviewer acknowledgments"
]

In [4]:
def is_valid_nlp_paper(title):
    """Filters out empty items and administrative noise records."""
    if not title:
        return False
    
    title_lower = title.lower().strip()
    
    # Drop structural metadata rows
    if any(blacklisted_term in title_lower for blacklisted_term in ADMIN_BLACKLIST):
        return False
        
    # Exclude structural session headers or short non-paper blocks
    if len(title_lower.split()) < 3:
        return False
        
    return True

records_processed = 0

print(f"Beginning crawl from: {PAPERS_REPO_DIR}")

# Open output context in write-stream mode
with open(OUTPUT_JSONL_PATH, "w", encoding="utf-8") as jsonl_out:
    
    # Recursively descend into directory structure
    for root, dirs, files in os.walk(PAPERS_REPO_DIR):
        folder_name = os.path.basename(root).lower()
        
        # Enforce pure NLP track domain alignment
        if folder_name not in NLP_CONFERENCES:
            continue
            
        for file in tqdm(files, desc=f"Ingesting {folder_name.upper()} Archives"):
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        raw_data = json.load(f)
                        
                    # Standardize schema mappings (handles both list arrays and dictionary hashes)
                    papers_iterable = raw_data.values() if isinstance(raw_data, dict) else raw_data
                    
                    for paper in papers_iterable:
                        # Defensive copy to avoid mutating cache structures
                        if not isinstance(paper, dict):
                            continue
                            
                        title = paper.get("title", "")
                        
                        # Validate entry cleanliness
                        if not is_valid_nlp_paper(title):
                            continue
                            
                        # Fallback parsing calculation for publication year if missing inside payload properties
                        extracted_year = paper.get("year") or "".join(filter(str.isdigit, file))
                        
                        # Deep-copy everything dynamically and layer normalized metrics over it
                        comprehensive_record = {**paper}
                        
                        comprehensive_record["venue"] = folder_name.upper()
                        comprehensive_record["year"] = int(extracted_year) if extracted_year else paper.get("year")
                        
                        # Stream straight to disk line-by-line
                        jsonl_out.write(json.dumps(comprehensive_record, ensure_ascii=False) + "\n")
                        records_processed += 1
                        
                except Exception as e:
                    print(f"Skipping unreadable payload at path [{file}]: {e}")

print(f"\nCompleted! Generated {records_processed} total NLP records in: '{OUTPUT_JSONL_PATH}'")


Beginning crawl from: /Users/salehalizade/Desktop/AXIOM/paperlists


Ingesting ACL Archives: 100%|██████████| 5/5 [00:00<00:00,  9.14it/s]


Completed! Generated 23002 total NLP records in: 'axiom_nlp_comprehensive_dataset.jsonl'


In [5]:
# Read the shared dataset file straight into a dataframe matrix
df = pd.read_json(OUTPUT_JSONL_PATH, lines=True)

# 1. Milestone Integrity Auditing
print(f"Total Papers Compiled: {len(df)}")
if len(df) >= 10000:
    print("✅ Target Met! Sufficient density reached for semantic indexing.")
else:
    print("⚠️ Target Alert! Volume falls below 10k baseline.")

# 2. Key Target Key Fields Integrity Checklist
critical_fields = ['id', 'author', 'bibtex', 'pdf', 'aff', 'github']
print("\n--- Feature Presence Verification ---")
for field in critical_fields:
    if field in df.columns:
        filled_percentage = (df[field].dropna() != "").sum() / len(df) * 100
        print(f"Field '{field}': Found in dataset (Data density: {filled_percentage:.2f}%)")
    else:
        print(f"Field '{field}': ❌ Missing from generated records")

# 3. Final Review of all extracted parameters
print("\n--- Extracted Data Sample ---")
df.head(2)

Total Papers Compiled: 23002
✅ Target Met! Sufficient density reached for semantic indexing.

--- Feature Presence Verification ---
Field 'id': Found in dataset (Data density: 100.00%)
Field 'author': Found in dataset (Data density: 100.00%)
Field 'bibtex': Found in dataset (Data density: 99.99%)
Field 'pdf': Found in dataset (Data density: 91.25%)
Field 'aff': Found in dataset (Data density: 86.48%)
Field 'github': Found in dataset (Data density: 30.78%)

--- Extracted Data Sample ---


,id,title,track,status,award,abstract,author,authorids,bibtex,pdf,...,confidence_avg,excitement_avg,reproducibility_avg,correctness_avg,replies_avg,authors#_avg,corr_rating_confidence,corr_rating_correctness,orcid,linkedin
0,2020.coling-main.519,"100,000 Podcasts: A Spoken English Document Co...",main,Main,0.0,Podcasts are a large and growing repository of...,Ann Clifton; Sravana Reddy; Yongze Yu; Aasish ...,/a/ann-clifton/; /s/sravana-reddy/; /y/yongze-...,"@inproceedings{clifton-etal-2020-100000,\n ...",https://aclanthology.org/2020.coling-main.519.pdf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020.coling-main.113,A BERT-based Dual Embedding Model for Chinese ...,main,Main,0.0,Chinese idioms are special fixed phrases usual...,Minghuan Tan; Jing Jiang,/m/minghuan-tan/; /j/jing-jiang/,"@inproceedings{tan-jiang-2020-bert,\n title...",https://aclanthology.org/2020.coling-main.113.pdf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.info()

In [ ]:
df.head(1)